In [ ]:
# Runtime setup for the Chapter 8 Cirq listings.
# The classifier snippet assumes these small training arrays already exist.
import numpy as np

x_train = np.array([
    [0.25, -0.35],
    [0.45, 0.30],
    [-0.40, 0.50],
    [-0.65, -0.20],
    [0.85, -0.75],
    [0.70, 0.65],
    [-0.80, 0.90],
    [-0.90, -0.55],
], dtype=float)
y_train = (x_train[:, 0] * x_train[:, 1] < 0.0).astype(float)

In [ ]:
import cirq
import sympy

qubits = cirq.LineQubit.range(2)
x0, x1, theta = sympy.symbols("x0 x1 theta")

circuit = cirq.Circuit()
circuit.append(cirq.rx(x0)(qubits[0]))
circuit.append(cirq.ry(x1)(qubits[1]))
circuit.append(cirq.CNOT(qubits[0], qubits[1]))
circuit.append(cirq.rz(theta)(qubits[1]))
circuit.append(cirq.measure(*qubits, key="z"))

resolver = {x0: 0.2, x1: -0.4, theta: 0.7}
sampler = cirq.Simulator(seed=123)
samples = sampler.run(circuit, param_resolver=resolver, repetitions=1000)
print(samples.histogram(key="z"))

In [ ]:
def encoding_block(qubits, features, layer_scale=1.0):
    for index, qubit in enumerate(qubits):
        yield cirq.rx(layer_scale * features[index % len(features)])(qubit)
        yield cirq.rz(layer_scale * features[(index + 1) % len(features)])(qubit)

def trainable_block(qubits, symbols):
    for qubit, symbol in zip(qubits, symbols):
        yield cirq.ry(symbol)(qubit)
    for left, right in zip(qubits[:-1], qubits[1:]):
        yield cirq.CNOT(left, right)

def reuploading_circuit(qubits, feature_symbols, weight_symbols, layers):
    circuit = cirq.Circuit()
    for layer in range(layers):
        circuit.append(encoding_block(qubits, feature_symbols, layer_scale=layer + 1))
        start = layer * len(qubits)
        stop = start + len(qubits)
        circuit.append(trainable_block(qubits, weight_symbols[start:stop]))
    return circuit

In [ ]:
class ZZFeatureGate(cirq.Gate):
    def __init__(self, angle):
        self.angle = angle

    def _num_qubits_(self):
        return 2

    def _decompose_(self, qubits):
        left, right = qubits
        yield cirq.CNOT(left, right)
        yield cirq.rz(2.0 * self.angle)(right)
        yield cirq.CNOT(left, right)

    def _circuit_diagram_info_(self, args):
        return ("ZZf", "ZZf")

q0, q1 = cirq.LineQubit.range(2)
phi = sympy.Symbol("phi")
circuit = cirq.Circuit(ZZFeatureGate(phi).on(q0, q1))

In [ ]:
theta = sympy.Symbol("theta")
qubit = cirq.LineQubit(0)

circuit = cirq.Circuit(
    cirq.rx(0.7)(qubit),
    cirq.ry(theta)(qubit),
    cirq.measure(qubit, key="z"),
)

sweep = cirq.Linspace(key=theta.name, start=-3.14, stop=3.14, length=25)
simulator = cirq.Simulator(seed=7)
results = simulator.run_sweep(circuit, params=sweep, repetitions=500)

expectations = []
for result in results:
    bits = result.measurements["z"][:, 0]
    expectations.append(1.0 - 2.0 * bits.mean())

In [ ]:
# Support objects used by the noisy-simulation listing.
# Recreate an unmeasured two-qubit circuit because the sweep example above
# binds the name `circuit` to a measured one-qubit diagnostic circuit.
qubits = cirq.LineQubit.range(2)
x0, x1, theta = sympy.symbols("x0 x1 theta")

circuit = cirq.Circuit()
circuit.append(cirq.rx(x0)(qubits[0]))
circuit.append(cirq.ry(x1)(qubits[1]))
circuit.append(cirq.CNOT(qubits[0], qubits[1]))
circuit.append(cirq.rz(theta)(qubits[1]))

resolver = {x0: 0.2, x1: -0.4, theta: 0.7}

In [ ]:
def z_expectation_from_samples(result, key="z"):
    bits = result.measurements[key][:, 0]
    return 1.0 - 2.0 * bits.mean()

def noisy_z_expectation(circuit, qubit, resolver, noise_probability, shots=2000):
    measured = circuit + cirq.measure(qubit, key="z")
    noisy_circuit = measured.with_noise(cirq.depolarize(noise_probability))
    simulator = cirq.DensityMatrixSimulator(seed=11)
    result = simulator.run(noisy_circuit, param_resolver=resolver, repetitions=shots)
    return z_expectation_from_samples(result)

noise_grid = [0.0, 0.001, 0.005, 0.01, 0.02]
curve = [
    noisy_z_expectation(circuit, qubits[0], resolver, noise_probability=p)
    for p in noise_grid
]

In [ ]:
import cirq
import numpy as np
import sympy
from scipy.optimize import minimize

qubits = cirq.LineQubit.range(2)
feature_symbols = sympy.symbols("x0 x1")
weight_symbols = sympy.symbols("theta0 theta1 theta2 theta3")

def make_model_circuit():
    circuit = cirq.Circuit()
    x0, x1 = feature_symbols
    t0, t1, t2, t3 = weight_symbols
    circuit.append([cirq.rx(x0)(qubits[0]), cirq.rx(x1)(qubits[1])])
    circuit.append(cirq.CNOT(qubits[0], qubits[1]))
    circuit.append([cirq.ry(t0)(qubits[0]), cirq.ry(t1)(qubits[1])])
    circuit.append(cirq.CNOT(qubits[1], qubits[0]))
    circuit.append([cirq.rz(t2)(qubits[0]), cirq.rz(t3)(qubits[1])])
    return circuit

model_circuit = make_model_circuit()
observable = cirq.Z(qubits[0])
simulator = cirq.Simulator(seed=19)

def score(features, weights):
    resolver = dict(zip(feature_symbols, features))
    resolver.update(dict(zip(weight_symbols, weights)))
    result = simulator.simulate(model_circuit, param_resolver=resolver)
    qubit_map = {qubit: index for index, qubit in enumerate(qubits)}
    return float(observable.expectation_from_state_vector(
        result.final_state_vector,
        qubit_map=qubit_map,
    ).real)

def probability(features, weights):
    return 0.5 * (1.0 - score(features, weights))

def binary_cross_entropy(weights, x_train, y_train):
    probabilities = np.array([probability(x, weights) for x in x_train])
    probabilities = np.clip(probabilities, 1e-6, 1.0 - 1e-6)
    return -np.mean(
        y_train * np.log(probabilities)
        + (1 - y_train) * np.log(1 - probabilities)
    )

initial_weights = np.zeros(len(weight_symbols))
result = minimize(
    binary_cross_entropy,
    initial_weights,
    args=(x_train, y_train),
    method="Nelder-Mead",
)

In [ ]:
def parameter_shift_gradient(features, weights, parameter_index):
    shift = np.zeros_like(weights)
    shift[parameter_index] = np.pi / 2.0
    plus = score(features, weights + shift)
    minus = score(features, weights - shift)
    return 0.5 * (plus - minus)

def gradient_vector(features, weights):
    return np.array([
        parameter_shift_gradient(features, weights, index)
        for index in range(len(weights))
    ])